# Experimentos ViDDAR — avaliação com VLMs locais

Este notebook roda a reprodução da avaliação do artigo ViDDAR (Xiu et al., IEEE TVCG 2025) e ajuda a
analisar os resultados, com foco nos **falsos positivos**. Ele usa as mesmas funções dos scripts em
`scripts/`, então os resultados são idênticos aos da linha de comando.

**Antes de começar**

- Selecione o kernel **"Python (ViDDAR .venv)"** (no VS Code: canto superior direito do notebook).
  Instalação do ambiente: README, seção 5.
- Rode as células em ordem. A configuração fica na seção 2.
- Respostas do VLM ficam em cache (`results/cache/`): repetir uma execução com o mesmo modelo e
  prompt não consulta o modelo de novo, só refaz detecção, segmentação e métricas.
- Tempo aproximado de uma execução nova na RTX 3060, dataset completo: obstrução com todos os
  métodos ~30–60 min; manipulação ~25 min (Qwen3-VL-2B) a ~40 min (modelos maiores).
- Para um teste rápido, use `LIMIT = 10`. Os resultados vão para `results/_smoke/` e não sobrescrevem
  os resultados completos.

## 1. Ambiente

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

# O notebook pode ser aberto a partir da raiz ou da pasta notebooks/.
ROOT = Path.cwd() if (Path.cwd() / "viddar").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display

from viddar import reporting, viz
from viddar.experiments import (ManipulationConfig, ObstructionConfig, load_run, manipulation_table,
                                obstruction_table, run_manipulation, run_obstruction)
from viddar.metrics import flatten, metrics_by_group
from viddar.vlm import PRESETS

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.max_rows", 60)

print("raiz do repositório:", ROOT)
print("Python:", sys.executable)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "não encontrada (só os métodos sem VLM vão rodar)")
print(f"disco livre: {shutil.disk_usage(ROOT).free / 1e9:.1f} GB")

## 2. Configuração

Edite e rode esta célula antes das seções seguintes.

In [ ]:
# VLM: um preset ou qualquer id do Hugging Face compatível com AutoModelForImageTextToText
print("presets:", list(PRESETS))
VLM = "qwen3-vl-2b"

LIMIT = None        # None = dataset completo; ex.: 10 para teste rápido
ALPHA = 0.25        # limiar da Eq. 4 (artigo: 0,25)
PROMPT = "paper"    # "paper" (prompts literais do artigo) ou "notebook" (prompts dos notebooks originais)

MANIFEST_OBSTRUCTION = "data/manifests/viddar_obstruction.csv"
MANIFEST_MANIPULATION = "data/manifests/viddar_manipulation.csv"

OUT_ROOT = "results" if LIMIT is None else "results/_smoke"
print("resultados em:", OUT_ROOT)

## 3. Preparar o dataset

Gera os manifestos e as máscaras do conteúdo virtual (só na primeira vez) e mostra o relatório de
validação do dataset. Requer o dataset original em `ViDDAR dataset/ViDDAR Dataset/` (README, seção 5).

In [ ]:
needs_prepare = not (Path(MANIFEST_OBSTRUCTION).exists() and Path(MANIFEST_MANIPULATION).exists()
                     and Path("data/content_masks/obstruction").exists())
if needs_prepare and MANIFEST_OBSTRUCTION.startswith("data/manifests/viddar_"):
    subprocess.run([sys.executable, "scripts/prepare_viddar.py"], check=True)

report = json.loads(Path("data/manifests/validation_report.json").read_text())
display(pd.json_normalize(report).T.rename(columns={0: "valor"}))

## 4. Detecção de obstrução (Tabela 1 do artigo)

Métodos disponíveis:

| Método | O que faz | Usa VLM | Usa GPU |
|---|---|---|---|
| `viddar` | VLM nomeia o objeto-chave → Grounding DINO + SAM → Eq. 4 | sim | sim |
| `end_to_end` | VLM nomeia o objeto e decide sozinho se ele está obstruído | sim | sim |
| `underdetailed` | como o ViDDAR, com prompt mínimo | sim | sim |
| `greedy` | VLM lista vários objetos; obstrução se qualquer um passar de α | sim | sim |
| `prior` | o detector recebe o nome correto do objeto (limite superior) | não | sim |
| `saliency`, `canny` | baselines clássicos de visão computacional | não | não |

In [ ]:
OBSTRUCTION_METHODS = ["viddar", "end_to_end", "underdetailed", "greedy", "prior", "saliency", "canny"]

obs_cfg = ObstructionConfig(manifest=MANIFEST_OBSTRUCTION, methods=OBSTRUCTION_METHODS, vlm=VLM,
                            alpha=ALPHA, key_prompt=PROMPT, limit=LIMIT, out=f"{OUT_ROOT}/obstruction")
obs_results = run_obstruction(obs_cfg)
obs_table = obstruction_table(obs_results)
display(obs_table.round(4))

Números do artigo para comparação (acurácia em %):

In [ ]:
display(reporting.PAPER_TABLE1)

### 4.1 Sensibilidade ao limiar α

A razão |m_k ∩ m_c| / |m_k| é salva por amostra, então a acurácia para qualquer α é recalculada
sem rodar o modelo. Com as máscaras verdadeiras, os rótulos do dataset se explicam melhor com α ≈ 0,2.

In [ ]:
runs_with_ratio = {m: r.predictions for m, r in obs_results.items() if "ratio" in r.predictions}
fig, sweep = viz.alpha_sweep(runs_with_ratio, paper_alpha=ALPHA)
plt.show()
display(sweep.set_index("alpha").round(3))

### 4.2 Onde estão os erros

Escolha um método e veja os falsos positivos e negativos, o desempenho por grupo (classe do
objeto-chave) e as respostas mais frequentes do VLM. Respostas repetidas que não correspondem à cena
(por exemplo "red box", copiado do exemplo do prompt) indicam problema de prompt.

In [ ]:
METHOD = "viddar"

pred = obs_results[METHOD].predictions
fp = pred[(pred["label"] == 0) & (pred["pred"] == 1)]
fn = pred[(pred["label"] == 1) & (pred["pred"] == 0)]
print(f"{METHOD}: {len(fp)} falsos positivos, {len(fn)} falsos negativos, {len(pred)} amostras")

cols = [c for c in ["id", "key_object", "key_phrase", "ratio", "key_iou", "n_detections"] if c in pred.columns]
display(Markdown("**Falsos positivos**"))
display(fp[cols])
display(Markdown("**Desempenho por grupo** (ordenado por número de falsos positivos)"))
display(metrics_by_group(pred).round(3))
if "key_phrase" in pred.columns:
    display(Markdown("**Respostas mais frequentes do VLM**"))
    display(pred["key_phrase"].value_counts().head(10).rename("ocorrências"))

Visualização de casos. Verde: objeto-chave verdadeiro. Laranja: conteúdo virtual. O título mostra o
que o VLM respondeu, a razão de obstrução e o IoU entre a máscara prevista e a verdadeira.

In [ ]:
N_SHOW = 3
for _, row in fp.head(N_SHOW).iterrows():
    viz.show_obstruction(row)
    plt.show()

In [ ]:
for _, row in fn.head(N_SHOW).iterrows():
    viz.show_obstruction(row)
    plt.show()

## 5. Detecção de manipulação de informação (Tabela 2 do artigo)

O VLM recebe a imagem crua e a aumentada com o questionário de seis perguntas do artigo. São
calculadas duas decisões:

- **regra do artigo:** o "yes"/"no" que aparece por último na resposta (decisão oficial, `pred`);
- **A ∧ S ∧ I:** combinação das respostas às perguntas 3, 4 e 5 (Eq. 7), para checar se a conclusão
  do VLM é coerente com as próprias respostas.

In [ ]:
man_cfg = ManipulationConfig(manifest=MANIFEST_MANIPULATION, vlm=VLM, prompt=PROMPT, limit=LIMIT,
                             out=f"{OUT_ROOT}/manipulation")
man_result = run_manipulation(man_cfg)
display(manipulation_table(man_result).round(4))

s = man_result.summary
print(f"conclusão coerente com as respostas 3-5: {s['final_answer_consistent_with_factors']:.1%}")
print(f"respostas sem os fatores interpretáveis: {s['unparsed_factor_answers']}")
display(Markdown("Números do artigo:"))
display(reporting.PAPER_TABLE2)

### 5.1 Qual fator causa os erros

Compara a proporção de "yes" nos rótulos com a do VLM para cada fator. Um VLM que diz "yes" ao fator I
muito mais do que o rótulo tende a produzir falsos positivos; um que diz muito menos, falsos negativos.

In [ ]:
mp = man_result.predictions
fig, rates = viz.factor_rates(mp)
plt.show()
display(rates.round(3))

display(Markdown("**Regra do artigo × A ∧ S ∧ I** (linhas: A ∧ S ∧ I; colunas: regra do artigo)"))
display(pd.crosstab(mp["pred_factor_rule"], mp["pred"], dropna=False))

In [ ]:
N_SHOW = 2
man_fp = mp[(mp["label"] == 0) & (mp["pred"] == 1)]
print(f"{len(man_fp)} falsos positivos")
for _, row in man_fp.head(N_SHOW).iterrows():
    viz.show_manipulation(row)
    plt.show()
    print("-" * 110)

## 6. Reabrir uma execução anterior

Para analisar resultados já salvos sem rodar nada, carregue a pasta da execução. Passando o manifesto,
os caminhos das imagens são recuperados e as funções de visualização funcionam normalmente.

In [ ]:
previous = load_run("results/manipulation/llava-next-8b-4bit/paper", manifest=MANIFEST_MANIPULATION)
print(previous.run_dir)
display(previous.predictions[["id", "label", "pred", "pred_A", "pred_S", "pred_I"]].head())

## 7. Todas as execuções lado a lado

Junta todas as execuções em `results/` (tabelas iguais às de `results/summary.md`). Valores em %.

In [ ]:
display(Markdown("**Obstrução**"))
display(reporting.collect_obstruction(Path(OUT_ROOT)))
display(Markdown("**Manipulação de informação**"))
display(reporting.collect_manipulation(Path(OUT_ROOT)))

## 8. Usando outro dataset

1. Crie um manifesto CSV (formato no README, seção 7). Use a coluna `group` para marcar os cenários do
   estudo, por exemplo casos ambíguos e casos de controle.
2. Aponte `MANIFEST_OBSTRUCTION` ou `MANIFEST_MANIPULATION` para ele na seção 2.
3. Rode as seções 4 e 5. A tabela "Desempenho por grupo" mostra a taxa de falsos positivos de cada
   cenário.